### Tool Subclassing BaseTool - Practice & Exercises

In this notebook, you will practice subclassing the abstract `BaseTool` class to implement advanced validation and execution workflows.

#### Part 1: Follow-Along Practice

##### Goal / Question
Subclass `BaseTool` to build a Tavily search tool and a multiplier tool, bind them to an agent, and execute queries.

##### Sample Input
```python
"Multiply 10 and 20"
```

##### Sample Output
The product of 10.0 and 20.0 is 200.0

##### Plan / Steps
1. Define custom Pydantic schemas for search query and multiplication inputs.
2. Define classes inheriting from `BaseTool`.
3. Assign class properties `name`, `description`, and `args_schema`.
4. Implement execution logic by overriding the abstract `_run` method.
5. Initialize model using `init_chat_model("llama-3.3-70b-versatile", model_provider="groq")`.
6. Instantiate tool subclasses, assemble agent executor, and execute queries.

In [ ]:
import os
from typing import Type
from dotenv import load_dotenv
from langchain import hub
from langchain.agents import AgentExecutor, create_tool_calling_agent
from pydantic import BaseModel, Field
from langchain_core.tools import BaseTool
from langchain.chat_models import init_chat_model

load_dotenv()

class SimpleSearchInput(BaseModel):
    query: str = Field(description="should be a search query")

class MultiplyNumbersArgs(BaseModel):
    x: float = Field(description="First number to multiply")
    y: float = Field(description="Second number to multiply")

class SimpleSearchTool(BaseTool):
    name = "simple_search"
    description = "useful for when you need to answer questions about current events"
    args_schema: Type[BaseModel] = SimpleSearchInput

    def _run(self, query: str) -> str:
        from tavily import TavilyClient
        api_key = os.getenv("TAVILY_API_KEY")
        client = TavilyClient(api_key=api_key)
        results = client.search(query=query)
        return f"Search results for: {query}\n\n{results}\n"

class MultiplyNumbersTool(BaseTool):
    name = "multiply_numbers"
    description = "useful for multiplying two numbers"
    args_schema: Type[BaseModel] = MultiplyNumbersArgs

    def _run(self, x: float, y: float) -> str:
        return f"The product of {x} and {y} is {x * y}"

tools = [SimpleSearchTool(), MultiplyNumbersTool()]
llm = init_chat_model("llama-3.3-70b-versatile", model_provider="groq")
prompt = hub.pull("hwchase17/openai-tools-agent")
agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)
executor = AgentExecutor.from_agent_and_tools(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)
print(executor.invoke({"input": "Multiply 10 and 20"}))

#### Part 2: Practice Exercises

##### Exercise: Temperature Converter Tool

##### Goal / Question
Define a custom tool `CelsiusToFahrenheit` by subclassing `BaseTool` that converts Celsius floats to Fahrenheit floats.

##### Sample Input
```python
{"input": "Convert 25 degrees Celsius to Fahrenheit"}
```

##### Sample Output
25.0 degrees Celsius is equal to 77.0 degrees Fahrenheit.

##### Plan / Steps
1. Define a Pydantic input validation model `TempConverterInput` with a float field `celsius`.
2. Subclass `BaseTool` to define `CelsiusToFahrenheitTool`.
3. Assign class properties and implement `_run(self, celsius: float) -> str` returning the conversion string ($F = C * 9/5 + 32$).
4. Append to tools list, rebuild agent, and test.

In [ ]:
# TODO: Write your implementation here

<details>
<summary>💡 Click to see solution</summary>

```python
class TempConverterInput(BaseModel):
    celsius: float = Field(description="Temperature in Celsius")

class CelsiusToFahrenheitTool(BaseTool):
    name = "celsius_to_fahrenheit"
    description = "Useful for converting Celsius temperatures to Fahrenheit."
    args_schema: Type[BaseModel] = TempConverterInput

    def _run(self, celsius: float) -> str:
        f = (celsius * 9/5) + 32
        return f"{celsius} degrees Celsius is equal to {f} degrees Fahrenheit."

extended_tools = tools + [CelsiusToFahrenheitTool()]
new_agent = create_tool_calling_agent(llm=llm, tools=extended_tools, prompt=prompt)
new_executor = AgentExecutor.from_agent_and_tools(agent=new_agent, tools=extended_tools, verbose=True, handle_parsing_errors=True)
print(new_executor.invoke({"input": "Convert 25 degrees Celsius to Fahrenheit"}))
```
</details>